# Download

Data Mining Cup 2016 order lines (fashion e-commerce, with return quantities).

**Setup.** `sys.path.append("..")` lets the notebook import from `src/` one level up. `RAW` resolves off the repo root rather than the working directory, so the path holds wherever this is run from.

In [1]:
import sys; sys.path.append("..")
import zipfile
import urllib.request
from src.data import RAW, load_orders

URL = "https://github.com/ISU-DMC/dmc2016/raw/master/data/raw_data/orders.zip"
ZIP = RAW / "orders.zip"

## Fetch

**Fetch and unpack.** Guarded by `if not ZIP.exists()`, so re-running the notebook never re-downloads 27MB. The data lands in `data/raw/`, which is gitignored — the DMC set is downloaded at runtime and never redistributed through this repo.

In [2]:
RAW.mkdir(parents=True, exist_ok=True)
if not ZIP.exists():
    urllib.request.urlretrieve(URL, ZIP)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(RAW)
sorted(p.name for p in RAW.iterdir())

['orders.zip', 'orders_class.txt', 'orders_train.txt']

## Check

**Load both files.** `orders_train` carries the `returnQuantity` label; `orders_class` is the unlabeled competition test set. Shapes here are a smoke test: if the mirror or file format ever changes, it fails at this cell rather than three notebooks later.

In [3]:
train, test = load_orders("orders_train"), load_orders("orders_class")
train.shape, test.shape

((2325165, 15), (341098, 14))

**Eyeball the raw grain.** One row per order *line*, not per order — the same `orderID` repeats across articles. That distinction drives the whole label decision in notebook 03.

In [4]:
train.head()

,orderID,orderDate,articleID,colorCode,sizeCode,productGroup,quantity,price,rrp,voucherID,voucherAmount,customerID,deviceID,paymentMethod,returnQuantity
0,a1000001,2014-01-01,i1000382,1972,44,3.0,1,10.00,29.99,0,0.0,c1010575,2,BPRG,0
1,a1000001,2014-01-01,i1000550,3854,44,3.0,1,20.00,39.99,0,0.0,c1010575,2,BPRG,0
2,a1000002,2014-01-01,i1001991,2974,38,8.0,1,35.00,49.99,0,0.0,c1045905,4,BPRG,0
3,a1000002,2014-01-01,i1001999,1992,38,8.0,1,49.99,49.99,0,0.0,c1045905,4,BPRG,1
4,a1000003,2014-01-01,i1001942,1968,42,8.0,1,10.00,35.99,0,0.0,c1089295,2,PAYPALVC,0


**Sanity check the span and scale.** Train covers Jan 2014 – Sep 2015 and test picks up immediately after, which is what makes a time-based split the honest choice. The line-level return rate lands at ~52%.

In [5]:
{"train dates": (train.orderDate.min().date(), train.orderDate.max().date()),
 "test dates": (test.orderDate.min().date(), test.orderDate.max().date()),
 "customers": train.customerID.nunique(),
 "articles": train.articleID.nunique(),
 "line return rate": (train.returnQuantity > 0).mean().round(4)}

{'train dates': (datetime.date(2014, 1, 1), datetime.date(2015, 9, 30)),
 'test dates': (datetime.date(2015, 10, 1), datetime.date(2015, 12, 31)),
 'customers': 311369,
 'articles': 3823,
 'line return rate': np.float64(0.5195)}